##### `Stape 1 : Prepare The Data`

In [ ]:
"""IMPORT EVERYTHING.. """
# Data manipulation
import pandas as pd
import numpy as np

# # Quantitative analysis / indicators
import talib
# import backtrader as bt
# import vectorbt as vbt


# # Machine learning / statistics
# import scipy
# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LinearRegression

# # Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import mplfinance as mpf


from resampler import OHLCVResampler
# # Jupyter Notebook magic for inline plots
# %matplotlib inline

import pandas_ta as ta



In [ ]:
"""Load Source OHLCV dta"""

df = pd.read_parquet("../Src/XAUUSDc_M1.parquet", engine="pyarrow")

In [ ]:
"""Setting Chart Configs"""
from ChartterX5 import Chartter

chartter = Chartter(config={
    'chart_type': 'candle',
    'style': 'charles',
    'volume': False
})

chartter._setup_tradingview_theme()

cnf = {
        'title': 'NONE',
        'style': chartter.tradingview_style,
        'volume': False,
        'show_nontrading': False,
        'datetime_format': '%Y-%m-%d %H:%M',
        'xlabel': '',
        'ylabel': '',
        'xrotation': 0
    }

resampler = OHLCVResampler(default_agg='last')

# result.head(3)

#### `lorem`

In [ ]:
dta = resampler.resample(df,'5min', '2024-10-16', '2024-10-17').ffill().copy()

dta['OC'] = ((dta['close'] - dta['open']) * 100 ).abs()

dta['oc_above_400'] = dta['OC'] > 200

mask = dta.index[dta['oc_above_400'] == True]


In [ ]:

fig, axes =chartter.plot(
    dta,
    config=cnf,
    # addplot=add_plots,
    vlines = {
        'vlines': mask.to_list(),
        'colors': 'orange',
        'linewidths': 0.9,
        'linestyle': '--',
        'alpha': 0.7
    },
    returnfig=True
)

In [ ]:
dta = resampler.resample(df,'1min', '2024-10-16', '2024-10-17').ffill()
dta = dta[:500]

In [ ]:
# --------------------------------+
# 3️⃣ Prepare data and indicators |
# ------------------------------ +
df = dta

# MACD
macd = ta.macd(df['close'], fastperiod=12, slowperiod=26, signalperiod=9)

df['MACD'] = macd['MACD_12_26_9']
df['MACD_signal'] = macd['MACDs_12_26_9']
df['MACD_hist'] = macd['MACDh_12_26_9']

df['MACD_Cross_up'] = ta.cross(df['MACD'],df['MACD_signal'])
df['MACD_Cross_dn'] = ta.cross(df['MACD_signal'],df['MACD'])

cross_up = df['MACD'].where(df['MACD_Cross_up'] != 0)
cross_dn = df['MACD'].where(df['MACD_Cross_dn'] != 0)

# RSI
df['RSI'] = ta.rsi(df['close'], length=14)
# df = jambura(df)


# df['signal'] = (df['MACD_Cross_dn'] == 1) & (df['signal_color'] == 'gray')
# df['signal'] = (df['MACD_Cross_dn'] == 1) & (df['signal_color'] == 'gray') & (df['RSI'] > 70 )

In [ ]:
import mplfinance as mpf
import matplotlib.dates as mdates
import pandas as pd

# Example dataframe
# df must have columns: 'Open', 'High', 'Low', 'Close'
# and a DateTime index
# df = your OHLC data (e.g. from yfinance)

# --- Addplots for MACD and RSI ---
add_plots = [
    mpf.make_addplot(df['MACD_hist'], panel=1, type='bar', color='dimgray', alpha=0.7),
    mpf.make_addplot(df['MACD_signal'], panel=1, type='line', color='green', alpha=0.7, width=0.8),
    mpf.make_addplot(df['MACD'], panel=1, type='line', color='blue', alpha=0.7, width=0.8),
    mpf.make_addplot(cross_up, panel=1, type='scatter', markersize=50, marker='o', color='yellow'),
    mpf.make_addplot(cross_dn, panel=1, type='scatter', markersize=50, marker='o', color='red'),

    mpf.make_addplot(df['RSI'], panel=2, type='line', color='orange', width=1),
    mpf.make_addplot(pd.Series(70, index=df.index), panel=2, type='line', color='red', width=0.5, alpha=0.5),
    mpf.make_addplot(pd.Series(30, index=df.index), panel=2, type='line', color='green', width=0.5, alpha=0.5),
]

# --- Base config ---
cnf = dict(
    type='candle',
    style='yahoo',          # You can also use 'charles', 'binance', etc.
    title='Hourly Grid Example',
    volume=False,
    datetime_format='%Y-%m-%d %H:%M',
    figsize=(21, 9),
    returnfig=True
)

# --- Create the plot ---
fig, axes = mpf.plot(df, addplot=add_plots, **cnf)

# --- Add hourly grid ---


# Finalize
import matplotlib.pyplot as plt
plt.show()
